# End-to-End Clinical Data-Flow on Synthetic Data

Demonstrates the complete privacy-preserving architecture pipeline:

| Stage | Method | Privacy property |
|---|---|---|
| 1 | Synthetic data generation | No real PHI at any point |
| 2 | FHE feature extraction | Raw feature values never leave encrypted domain |
| 3 | Differentially private training | Formal (ε, δ) bound on individual record influence |
| 4 | LLM leakage assessment | Automated detection of prompt-injection / log-capture / membership-inference |

> **No real patient data is used.** All values are synthetic and statistically plausible
> but do not correspond to real individuals.

**Federal alignment:** NIST AI RMF (GOVERN 1.1, MAP 2.1), NIST Privacy Framework
CT.PO-P3 / CT.DM-P8, CISA AI Data Security guidance (2024).

In [ ]:
"""Stage 0 — Environment setup and path configuration."""
import os
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Add module roots so imports work when notebook runs from examples/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for subdir in ["fhe-feature-extraction", "dp-llm-training", "llm-leakage-assessment"]:
    path = os.path.join(REPO_ROOT, subdir)
    if path not in sys.path:
        sys.path.insert(0, path)

np.random.seed(42)
print("Environment ready. Repo root:", REPO_ROOT)

## Stage 1 — Synthetic Clinical Dataset

512 synthetic records with 6 numerical features modelled on published population norms
(CDC NHANES summary statistics). Values are drawn from calibrated distributions;
they are not sampled from any real dataset.

In [ ]:
def generate_synthetic_clinical(
    n: int = 512,
    seed: int = 42,
) -> pd.DataFrame:
    """Reproducible synthetic clinical records. No real PHI used."""
    rng = np.random.default_rng(seed)
    systolic = rng.normal(120.0, 15.0, n).clip(80, 200)
    diastolic = rng.normal(80.0, 10.0, n).clip(50, 130)
    diastolic = np.minimum(diastolic, systolic - 10)  # enforce SBP > DBP

    return pd.DataFrame(
        {
            "age": rng.integers(18, 90, n),
            "systolic_bp": np.round(systolic, 1),
            "diastolic_bp": np.round(diastolic, 1),
            "glucose_mg_dl": np.round(rng.normal(100.0, 25.0, n).clip(60, 400), 1),
            "bmi": np.round(rng.normal(26.5, 5.0, n).clip(15, 55), 1),
            "heart_rate": rng.integers(40, 140, n),
            "label": rng.choice([0, 1], n, p=[0.65, 0.35]),
        }
    )


df = generate_synthetic_clinical(n=512, seed=42)
print(f"Dataset: {df.shape[0]} synthetic records x {df.shape[1] - 1} features")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")
df.describe().round(2)

## Stage 2 — FHE Feature Extraction

Features are extracted using the CKKS fully homomorphic encryption scheme.
The raw feature values are encrypted immediately; the linear projection
that maps 6 clinical dimensions to a 4-dimensional representation runs
entirely on ciphertext. Raw values are never exposed during extraction.

In [ ]:
from fhe_pipeline import DataMinimizationPipeline, FHEContext, FHEFeatureExtractor

INPUT_DIM = 6  # six clinical features
FEATURE_DIM = 4  # compact representation

print("Initialising FHE context (CKKS, poly_modulus_degree=8192)...")
ctx = FHEContext(poly_modulus_degree=8192)

extractor = FHEFeatureExtractor(
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    fhe_context=ctx,
)
pipeline = DataMinimizationPipeline(extractor)

feature_col_names = ["age", "systolic_bp", "diastolic_bp", "glucose_mg_dl", "bmi", "heart_rate"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_col_names].values)

# Extract encrypted features for first 8 records (demo subset)
fhe_features = []
for i in range(8):
    feats = pipeline.process(
        data_id=f"SYN-{i:06d}",
        plaintext_input=X_scaled[i],
        requester="analysis-node-A",
        purpose="feature extraction for downstream DP training demo",
    )
    fhe_features.append(feats)

print(f"Extracted encrypted features for {len(fhe_features)} records.")
print(f"Feature shape per record: {fhe_features[0].shape}")
print()
print("Audit log (raw_data_returned must always be False):")
for entry in pipeline.get_audit_log():
    print(
        f"  {entry['data_id']} | raw_data_returned={entry['raw_data_returned']} | "
        f"features_returned={entry['features_returned']}"
    )

# Verify round-trip accuracy
_, enc = extractor.extract(X_scaled[0])
recovered = extractor.decrypt_features(enc)
expected = extractor.W @ X_scaled[0]
max_err = np.max(np.abs(recovered - expected))
print(f"\nRound-trip reconstruction error (max): {max_err:.2e}  ", end="")
print("✓ PASS" if max_err < 1e-3 else "✗ FAIL")

## Stage 3 — Differentially Private Training

A binary classifier is trained on the full synthetic dataset using the
DPTrainer wrapper, which applies per-example gradient clipping and
calibrated Gaussian noise to achieve formal (ε, δ) privacy guarantees
(Opacus, RDP accountant).

> **Interpretation:** ε ≈ 3 means any single record's presence or absence
> changes the model's output distribution by at most a factor of e³ ≈ 20.
> δ = 1e-5 is the probability that this bound fails.

In [ ]:
import torch
from dp_trainer import DPTrainer, PrivacyConfig

# Prepare full dataset (plaintext features for DP training — FHE above was the
# extraction stage; DP is the training-time control)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["label"].values, test_size=0.2, random_state=42
)

config = PrivacyConfig(
    target_epsilon=3.0,
    target_delta=1e-5,
    max_grad_norm=1.0,
)

trainer = DPTrainer(
    input_dim=INPUT_DIM,
    hidden_dim=16,
    output_dim=2,
    privacy_config=config,
)

X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)

print(
    f"Training on {len(X_tr)} synthetic records with DP (target ε={config.target_epsilon}, δ={config.target_delta})\n"
)
NUM_EPOCHS = 3
for epoch in range(1, NUM_EPOCHS + 1):
    loss = trainer.step(X_tr, y_tr)
    budget = trainer.privacy_budget_spent()
    print(
        f"Epoch {epoch}/{NUM_EPOCHS}  "
        f"ε={budget['epsilon']:.3f}  "
        f"δ={budget['delta']:.0e}  "
        f"loss={loss:.4f}"
    )

final_budget = trainer.privacy_budget_spent()
print(
    f"\nFinal privacy budget consumed: ε={final_budget['epsilon']:.3f}, δ={final_budget['delta']:.0e}"
)
print(f"Budget within target (ε≤{config.target_epsilon}): ", end="")
print("✓ YES" if final_budget["epsilon"] <= config.target_epsilon else "✗ EXCEEDED")

## Stage 4 — LLM Leakage Assessment

The `AssessmentRunner` evaluates a set of YAML-defined test cases covering
the seven leakage threat categories in `threat-taxonomy.md`. In this notebook
it runs against the built-in `MockModel` (a safe, non-leaking stand-in)
to demonstrate the assessment framework. Replace `MockModel` with a real
local model to assess production systems.

In [ ]:
from assessment_runner import AssessmentRunner, MockModel

# Point test_cases_dir at the repo's test cases (relative to notebook location)
TEST_CASES_DIR = os.path.join(REPO_ROOT, "llm-leakage-assessment", "test_cases")

runner = AssessmentRunner(
    model=MockModel(),  # swap for a real model in production assessment
    test_cases_dir=TEST_CASES_DIR,
)

results = runner.run_all()

print("=== LLM Leakage Assessment Results ===")
print(f"Total test cases evaluated: {results['summary']['total']}")
print(f"Passed (no leakage detected):  {results['summary']['passed']}")
print(f"Failed (leakage signal found): {results['summary']['failed']}")
print()

# Show per-category breakdown
categories = {}
for case in results.get("results", []):
    cat = case.get("category", "unknown")
    categories.setdefault(cat, {"passed": 0, "failed": 0})
    if case["passed"]:
        categories[cat]["passed"] += 1
    else:
        categories[cat]["failed"] += 1

print(f"{'Category':<35} {'Passed':>7} {'Failed':>7}")
print("-" * 51)
for cat, counts in sorted(categories.items()):
    print(f"{cat:<35} {counts['passed']:>7} {counts['failed']:>7}")

overall = "LOW" if results["summary"]["failed"] == 0 else "ELEVATED"
print(f"\nOverall leakage risk: {overall}")

## Stage 5 — End-to-End Summary

Audit-ready confirmation that no raw PHI was exposed at any pipeline stage.

In [ ]:
import json
from datetime import datetime, timezone

summary = {
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "records": len(df),
        "features": INPUT_DIM,
        "synthetic_only": True,
        "real_phi_used": False,
    },
    "fhe_stage": {
        "scheme": "CKKS",
        "poly_modulus_degree": ctx.poly_modulus_degree,
        "records_processed": len(fhe_features),
        "raw_data_ever_returned": not all(
            not e["raw_data_returned"] for e in pipeline.get_audit_log()
        ),
        "max_reconstruction_error": float(max_err),
    },
    "dp_training_stage": {
        "epsilon_consumed": final_budget["epsilon"],
        "delta": final_budget["delta"],
        "target_epsilon": config.target_epsilon,
        "budget_within_target": final_budget["epsilon"] <= config.target_epsilon,
        "epochs": NUM_EPOCHS,
        "training_records": len(X_tr),
    },
    "leakage_assessment_stage": {
        "test_cases_run": results["summary"]["total"],
        "passed": results["summary"]["passed"],
        "failed": results["summary"]["failed"],
        "overall_risk": overall,
    },
}

print(json.dumps(summary, indent=2))
print("\n" + "=" * 60)
print("ARCHITECTURE DEMONSTRATION COMPLETE")
print("All three privacy controls verified on synthetic clinical data.")
print("No real PHI used or exposed at any pipeline stage.")
print("=" * 60)